# Load and Explore the Dataset

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Load the dataset
df = pd.read_csv('financial_anomaly_data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

# Data Preprocessing and Cleaning

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Remove duplicates
df = df.drop_duplicates()
print("\nShape after removing duplicates:", df.shape)

# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Encode categorical variables
le_merchant = LabelEncoder()
le_transaction_type = LabelEncoder()
le_location = LabelEncoder()

df['Merchant_encoded'] = le_merchant.fit_transform(df['Merchant'])
df['TransactionType_encoded'] = le_transaction_type.fit_transform(df['TransactionType'])
df['Location_encoded'] = le_location.fit_transform(df['Location'])

# Select numerical features for anomaly detection
numerical_features = ['Amount', 'Merchant_encoded', 'TransactionType_encoded', 'Location_encoded']
df_numerical = df[numerical_features]

# Scale the data
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_numerical), columns=numerical_features)

print("\nPreprocessed data shape:", df_scaled.shape)
print("Preprocessed data head:")
print(df_scaled.head())

# Exploratory Data Analysis

In [ ]:
# Distribution of Amount
plt.figure(figsize=(10, 6))
sns.histplot(df['Amount'], bins=50, kde=True)
plt.title('Distribution of Transaction Amounts')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.show()

# Box plot for Amount
plt.figure(figsize=(8, 6))
sns.boxplot(y=df['Amount'])
plt.title('Box Plot of Transaction Amounts')
plt.show()

# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df_scaled.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# Transaction types distribution
plt.figure(figsize=(10, 6))
df['TransactionType'].value_counts().plot(kind='bar')
plt.title('Distribution of Transaction Types')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.show()

# Amount by Transaction Type
plt.figure(figsize=(12, 8))
sns.boxplot(x='TransactionType', y='Amount', data=df)
plt.title('Transaction Amounts by Type')
plt.xticks(rotation=45)
plt.show()

# Feature Engineering

In [ ]:
# Extract time-based features
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek
df['Month'] = df['Timestamp'].dt.month

# Group by AccountID and calculate statistics
account_stats = df.groupby('AccountID')['Amount'].agg(['mean', 'std', 'count']).reset_index()
account_stats.columns = ['AccountID', 'Account_Mean_Amount', 'Account_Std_Amount', 'Account_Transaction_Count']

# Merge back to df
df = df.merge(account_stats, on='AccountID', how='left')

# Calculate deviation from account mean
df['Amount_Deviation_From_Mean'] = df['Amount'] - df['Account_Mean_Amount']

# Fill NaN for std (if count=1)
df['Account_Std_Amount'] = df['Account_Std_Amount'].fillna(0)

# Update numerical features
numerical_features.extend(['Hour', 'DayOfWeek', 'Month', 'Account_Mean_Amount', 'Account_Std_Amount', 'Account_Transaction_Count', 'Amount_Deviation_From_Mean'])

# Update scaled data
df_numerical = df[numerical_features]
df_scaled = pd.DataFrame(scaler.fit_transform(df_numerical), columns=numerical_features)

print("Updated features:", numerical_features)
print("Scaled data shape:", df_scaled.shape)

# Implement Isolation Forest Model

In [ ]:
# Train Isolation Forest
# Assuming contamination rate of 5% (adjust based on domain knowledge)
contamination_rate = 0.05

iso_forest = IsolationForest(n_estimators=100, contamination=contamination_rate, random_state=42)
iso_forest.fit(df_scaled)

# Predict anomalies
df['Anomaly_Score'] = iso_forest.decision_function(df_scaled)
df['Anomaly'] = iso_forest.predict(df_scaled)

# Convert predictions: -1 for anomaly, 1 for normal
df['Anomaly'] = df['Anomaly'].map({1: 0, -1: 1})  # 0: normal, 1: anomaly

print("Anomaly detection completed.")
print("Number of anomalies detected:", df['Anomaly'].sum())
print("Percentage of anomalies:", (df['Anomaly'].sum() / len(df)) * 100)

# Model Evaluation and Results

In [ ]:
# Since we don't have ground truth labels, we'll evaluate based on statistical analysis
# Analyze anomalies
anomalies = df[df['Anomaly'] == 1]
normal = df[df['Anomaly'] == 0]

print("Anomalies Statistics:")
print(anomalies['Amount'].describe())
print("\nNormal Transactions Statistics:")
print(normal['Amount'].describe())

# Check if anomalies have extreme values
print("\nTop 10 highest anomaly scores:")
print(df.nlargest(10, 'Anomaly_Score')[['Amount', 'TransactionType', 'Merchant', 'Anomaly_Score']])

# Distribution of anomaly scores
plt.figure(figsize=(10, 6))
sns.histplot(df['Anomaly_Score'], bins=50, kde=True)
plt.title('Distribution of Anomaly Scores')
plt.xlabel('Anomaly Score')
plt.ylabel('Frequency')
plt.show()

# Anomalies by transaction type
plt.figure(figsize=(12, 8))
anomalies['TransactionType'].value_counts().plot(kind='bar')
plt.title('Anomalies by Transaction Type')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.show()

# Visualize Anomalies

In [ ]:
# Scatter plot of Amount vs Anomaly Score
plt.figure(figsize=(12, 8))
plt.scatter(df['Amount'], df['Anomaly_Score'], c=df['Anomaly'], cmap='coolwarm', alpha=0.6)
plt.colorbar(label='Anomaly (1) vs Normal (0)')
plt.title('Transaction Amount vs Anomaly Score')
plt.xlabel('Amount')
plt.ylabel('Anomaly Score')
plt.show()

# Box plot comparison
plt.figure(figsize=(10, 6))
sns.boxplot(x='Anomaly', y='Amount', data=df)
plt.title('Amount Distribution: Normal vs Anomalies')
plt.xticks([0, 1], ['Normal', 'Anomaly'])
plt.show()

# Time series of anomalies
df_anomalies = df[df['Anomaly'] == 1].copy()
df_anomalies = df_anomalies.sort_values('Timestamp')

plt.figure(figsize=(15, 8))
plt.plot(df['Timestamp'], df['Amount'], alpha=0.5, label='Normal')
plt.scatter(df_anomalies['Timestamp'], df_anomalies['Amount'], color='red', label='Anomaly', s=50)
plt.title('Time Series of Transactions with Anomalies Highlighted')
plt.xlabel('Timestamp')
plt.ylabel('Amount')
plt.legend()
plt.show()

print("Analysis complete. The Isolation Forest model has identified potential anomalies in the financial transaction data.")

# Save Results for Web Dashboard

In [ ]:
# Save processed data and results
import pickle

# Save the DataFrame with results
df.to_csv('financial_anomaly_results.csv', index=False)

# Save the model and scaler
with open('isolation_forest_model.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Create summary statistics
summary = {
    'total_transactions': len(df),
    'anomalies_detected': int(df['Anomaly'].sum()),
    'anomaly_percentage': float((df['Anomaly'].sum() / len(df)) * 100),
    'avg_amount': float(df['Amount'].mean()),
    'avg_anomaly_amount': float(df[df['Anomaly'] == 1]['Amount'].mean()),
    'avg_normal_amount': float(df[df['Anomaly'] == 0]['Amount'].mean()),
}

import json
with open('summary_statistics.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Data saved successfully!")
print(f"- financial_anomaly_results.csv")
print(f"- isolation_forest_model.pkl")
print(f"- scaler.pkl")
print(f"- summary_statistics.json")